# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and performing basic processing on a FAIR-compliant dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/api_reference/python/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print out the dataset metadata
metadata = dataset.metadata

print(f"{metadata.name} (Version: {metadata.version})\nPublished: {metadata.datePublished}")
print("\nDescription:")
print(metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The Croissant schema specifies data structure in terms of record sets and fields. We list all record sets, their IDs, and the fields (columns) within each.

In [ ]:
# List all record sets and the fields for each with their @id
all_record_sets = dataset.metadata.record_sets
for rs in all_record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields in this record set:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print('-' * 60)

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame for further analysis.

Use the `@id` of each record set and its fields to reference them programmatically.

In [ ]:
# Build list of record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading data for record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  No records found for this record set.")
    print('-'*50)

# For demonstration, pick the first non-empty record set for deeper EDA
selected_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        selected_record_set_id = rsid
        break

if selected_record_set_id:
    print(f"Proceeding with record set: {selected_record_set_id}")
    print(dataframes[selected_record_set_id].head())
else:
    print("No record set with records found.")

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing operations such as filtering, normalization, and grouping on the chosen record set.

For demonstration, let's: 
- List all fields for the selected record set
- Select a numeric field (e.g., 'MW [kDa]' or another mass/abundance field)
- Filter proteins by a threshold on this field
- Normalize the field
- Group by another field (if applicable, e.g., by 'Sample' or 'Gene Name')

In [ ]:
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    print(f"Fields in record set '@id': {selected_record_set_id}\n{df.columns.tolist()}")

    # Try to automatically select a likely numeric field (fallback to first float column)
    possible_numeric_fields = [
        col for col in df.columns if any(s in col.lower() for s in ['mw', 'mass', 'abundance', 'count', 'coverage', 'peptide', 'score'])
    ]
    if not possible_numeric_fields:
        # If none matched by heuristic, just pick first numeric-looking
        possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
        if not possible_numeric_fields and len(df.columns) > 0:
            possible_numeric_fields = [df.columns[0]]

    numeric_field = possible_numeric_fields[0]
    print(f"Selected numeric field for demonstration: {numeric_field}")

    # Drop null values for the chosen numeric field and convert to float
    df_clean = df.dropna(subset=[numeric_field]).copy()
    df_clean[numeric_field] = pd.to_numeric(df_clean[numeric_field], errors='coerce')

    threshold = df_clean[numeric_field].mean()

    filtered_df = df_clean[df_clean[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}), count: {len(filtered_df)}")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to select a group field
    possible_group_fields = [
        col for col in df_clean.columns if any(s in col.lower() for s in ['sample', 'gene', 'condition', 'group'])
    ]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped mean {numeric_field} by {group_field} (first 5 groups):")
        print(grouped_df.head())
    else:
        print("No suitable categorical/grouping field found for demonstration.")
else:
    print("No EDA available; no record set with data.")

## 5. Visualization
Visualize the distribution of the selected numeric field and the grouped means, if grouping was possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field:
    # Histogram of numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df_clean[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped, show top-10 means by group
    if group_field:
        plt.figure(figsize=(8,4))
        grouped_df_sorted = grouped_df.sort_values(numeric_field, ascending=False).head(10)
        sns.barplot(x=grouped_df_sorted.index, y=grouped_df_sorted[numeric_field], palette='viridis')
        plt.title(f"Top 10 {group_field} by mean {numeric_field}")
        plt.xlabel(group_field)
        plt.xticks(rotation=45)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion

- We loaded the mass spectrometry dataset using the FAIR Croissant schema and the `mlcroissant` Python API.
- The data structure was explored by iterating over record sets and their fields by `@id`.
- We selected a numeric field and applied basic filtering, normalization, and grouping.
- Data visualizations illustrated distributions and group-level summaries.

This process provides a reproducible template for working with any well-described dataset in Croissant format. For advanced analyses, integrate domain-specific statistical and machine learning tools as needed!